# UI: Streamlit

The UI of this LLM app was developed with Streamlit, to keep things in Python.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from svsvllm.utils.nb import nb_init

nb_init()

INFO | nb_init | Set current dir to app
INFO | nb_init | You are using Python 3.12.7 (main, Nov 20 2024, 14:24:14) [Clang 16.0.0 (clang-1600.0.26.4)]


## Session sate

Streamlit session state works as a global container where you can store any object, so that these objects are available to other functions and/or parts of your source code.

Most Streamlit's components (like button, etc.) also end up in this session state.


In [2]:
import streamlit as st

for name, value in st.session_state.items():
    print(f"{name}: {value}")

2025-01-08 09:08:40.803 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


The drawbacks of this session state are the following:

- You never know if a certain key is available in the state and have to always check for its existence before using it;
- You have manually to remember the type of the value for each key, as the state is not typed and no type hints are possible (considering that the values can be anything).

Wouldn't it be more convenient if we could leverage something like `pydantic` for this session state?


### Typed session state

The present library has its own session state, which is a `pydantic.BaseModel` object, so that you know in advance what each key has to hold and whether it has a default value or not.

Not only, by using the `pydantic.BaseModel` class, you benefit from having field descriptions readily available and validation on key assignment. E.g., if a certain key is expected to hold a `str` object, you won't be allowed to pass a `float`.


In [3]:
from svsvchat.session_state import session_state

session_state

2025-01-08 09:14:05.073 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-08 09:14:05.074 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-08 09:14:05.074 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-08 09:14:05.074 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-08 09:14:05.075 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-08 09:14:05.075 WARNING streamlit.run

SessionState(state=None, reverse=True, auto_sync=False, streaming=True, chat_activated=False, language='English', new_language='English', openai_api_key=None, uploaded_files=[], callbacks={}, saved_filenames=[], page='main', openai_client=None, openai_model_name='gpt-3.5-turbo', model_name='mlx-community/quantized-gemma-2b-it', embedding_model_name='BAAI/bge-base-en-v1.5', chunk_size=512, chunk_overlap=30, model=None, tokenizer=None, db=None, retriever=None, history_aware_retriever=None, chat_model=None, chat_history=[], agent=None, thread_id=None, agent_config={}, quantize=True, quantize_w_torch=True, use_mlx=True, use_react_agent=True)

You can bind this session state to Streamlit's:


In [4]:
session_state.bind()  # or more explicitly: session_state.bind(st.session_state)

And now you can work with this typed container instead of `st.session_state`.

The two objects are bound, meaning that adding an objet to one will also add it to the other.


In [5]:
st.session_state["chunk_size"] = 2

2025-01-08 09:17:08.299 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-08 09:17:08.300 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [6]:
session_state.chunk_size

2

By default, the sync is one-direction. If you update `session_state` (not `st.session_state`), then you have to call `manual_sync()` (unless you set `session_state.auto_sync = True`):


In [10]:
session_state.chunk_overlap = 2
session_state.manual_sync("chunk_overlap")

2025-01-08 09:18:28.464 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-08 09:18:28.465 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-08 09:18:28.466 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-08 09:18:28.472 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-01-08 09:18:28.476 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


In [11]:
st.session_state["chunk_overlap"]

2025-01-08 09:18:30.207 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2